# Predicting Life Expectancy from Health & Economic Indicators

This project predicts a country's life expectancy from health, economic, and
social indicators, and compares four regression models. The best model
(Random Forest) explains about 96% of the variance in life expectancy on
held-out data.

The analysis shows that HIV/AIDS prevalence and income composition of resources account for most of the model's decisions.

**Dataset:** a WHO-derived set of 2,938 country-year records (2000-2015) with 22
indicators and a single life-expectancy target (see the README for provenance).

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import os
os.makedirs('../images', exist_ok=True)   # plots are saved into the repo-root images/ folder

import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.expand_frame_repr', False)

## 1. Data understanding and exploration

I start by analyzing the data distributions and temporal patterns before
modeling. This helps identify data quality issues like missing values or
outliers that could hurt model performance, reveals relationships between the
features and the target, and flags anomalies early so the model doesn't learn
from unusual events that don't represent normal patterns.

In [ ]:
life_exp_df = pd.read_csv('life_expectancy.csv')
print(f'Dataset shape: {life_exp_df.shape}')
print(f'First 5 rows:')
life_exp_df.head()

In [ ]:
print(f'Summary statistics of the dataset:')
life_exp_df.describe()

In [ ]:
print(f'Dataset information:')
life_exp_df.info()

In [ ]:
# Identifying categorical and numerical columns
print(f"\nColumn Names and Types:")
print(life_exp_df.dtypes)

categorical_cols = life_exp_df.select_dtypes(include=['object']).columns.tolist()
numerical_cols = life_exp_df.select_dtypes(include=[np.number]).columns.tolist()

print(f"Categorical Columns ({len(categorical_cols)}): {categorical_cols}")
print(f"Numerical Columns ({len(numerical_cols)}): {numerical_cols}")

The dataset contains 2,938 observations across 22 variables, with 2 categorical
variables (Country and Status) and 20 numerical ones. Each row represents one
country in one year, so there are multiple years per country. Life expectancy
ranges from 36.3 to 89 years, with a mean of about 69 years. Adult Mortality is
highly variable (1 to 723 per 1,000), likely driven by the difference between
developed and developing countries and by health crises like HIV/AIDS.

### Average life expectancy by status

In [ ]:
average_life_exp = life_exp_df.groupby(['Year','Status'])['Life expectancy '].mean().reset_index()
print('Average Life Expectancy for developed and developing countries: \n')
average_life_exp

### Life expectancy trend: developed vs developing

In [ ]:
df_2010_to_2015 = life_exp_df[(life_exp_df['Year'] >= 2000) & (life_exp_df['Year'] <= 2015)]

year_avg = df_2010_to_2015.groupby(['Year', 'Status'])['Life expectancy '].mean().reset_index()

developed_countries = year_avg[year_avg['Status'] == 'Developed']
developing_countries = year_avg[year_avg['Status'] == 'Developing']

plt.figure(figsize=(12, 6))
plt.plot(developed_countries['Year'], developed_countries['Life expectancy '], marker='o', linewidth=1, label='Developed Countries', color='blue')
plt.plot(developing_countries['Year'], developing_countries['Life expectancy '], marker='o', linewidth=1, label='Developing Countries', color='green')
plt.xlabel('Year')
plt.ylabel('Average Life Expectancy')
plt.title('Developed vs Developing Countries Life Expectancy Trends (2000-2015)')
plt.xticks(range(2000, 2016, 1), rotation=45)
plt.legend()
plt.tight_layout()
plt.savefig('../images/life_expectancy_trend.png', dpi=120, bbox_inches='tight')
plt.show()

Both groups show gradual increases over the 15-year period, but with a
consistent gap of about 10-15 years between them. There is a small anomaly in
2008 for developed countries, where life expectancy dips instead of rising,
which aligns with reports of a recession-year drop in some developed countries.
The gap between the two groups suggests that country Status will matter for
modeling.

## 2. Handling missing values

I examine the missing values per column, then compare two imputation methods on
GDP (the variable with the second-highest number of missing values): filling
with the median, versus KNN imputation.

In [ ]:
print(f"\nMissing Values:")
missing_values = life_exp_df.isnull().sum()
print(missing_values)

In [ ]:
missing_values = life_exp_df.isnull().sum()
missing_proportions = missing_values / len(life_exp_df)
missing_percentages = missing_proportions * 100

missing_values_df = pd.DataFrame({
    'Missing Values Count': missing_values,
    'Proportion': missing_proportions,
    'Percentage (%)': missing_percentages
})

print("\nMissing Values Summary:")
missing_values_df

Several variables have substantial gaps, with Population, Hepatitis B, and GDP
the most affected. I compare median imputation against KNN imputation below.

In [ ]:
# Original dataset
df_original = life_exp_df.copy()

df_median = df_original.copy()
median_val = df_median['GDP'].median()
missing_count = df_median['GDP'].isnull().sum()
df_median['GDP'] = df_median['GDP'].fillna(median_val)
print(f" \n {missing_count} ({missing_count/len(life_exp_df)*100:.2f} %) Missing values filled with  '{median_val}'")

In [ ]:
# Create KNNImputer instance with 2 nearest neighbors
df_knn = df_original.copy()
original_mean = df_original['GDP'].mean()

imputer = KNNImputer(n_neighbors=2)
missing_count_knn = df_knn['GDP'].isna().sum()
df_knn[['GDP']] = imputer.fit_transform(df_knn[['GDP']])

new_mean = df_knn['GDP'].mean()
print(f"KNN Imputation: {missing_count_knn} values filled (values vary for each missing entry)")

In [ ]:
stats = pd.DataFrame({
    'Original': df_original['GDP'].describe(),
    'Median Imputed': df_median['GDP'].describe(),
    'KNN Imputed': df_knn['GDP'].describe()
})
print(stats[['Original','Median Imputed','KNN Imputed']])

In [ ]:
# Creating subplots
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Plot 1: Original (with missing values removed for visualization)
axes[0].hist(np.log1p(df_original['GDP']).dropna(), bins=30, color='blue', edgecolor='black', alpha=0.7)
axes[0].set_title('Original GDP\n(Missing values excluded)')
axes[0].set_xlabel('GDP')
axes[0].set_ylabel('Frequency')
axes[0].grid(alpha=0.1)

# Plot 2: After Median Imputation
axes[1].hist(np.log1p(df_median['GDP']), bins=30, color='coral', edgecolor='black', alpha=0.7)
axes[1].axvline(np.log1p(median_val), color='red', linestyle='--', linewidth=2,
                label=f'Median: {median_val:.2f}')
axes[1].set_title('After Median Imputation')
axes[1].set_xlabel('GDP')
axes[1].set_ylabel('Frequency')
axes[1].legend()
axes[1].grid(alpha=0.1)

# Plot 3: After KNN Imputation
axes[2].hist(np.log1p(df_knn['GDP']), bins=30, color='teal', edgecolor='black', alpha=0.7)
axes[2].set_title('After KNN Imputation')
axes[2].set_xlabel('GDP')
axes[2].set_ylabel('Frequency')
axes[2].grid(alpha=0.1)

plt.tight_layout()
plt.savefig('../images/imputation_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

Median imputation is computationally efficient and preserves the median by
definition, but it reduces variability (all imputed values are identical) and
ignores relationships between GDP and other variables. The KNN method preserved
the original mean of 7483.16, while median imputation reduced it to 6611.52, and
the distribution after KNN imputation looked more natural and similar to the
original data. So I use KNN imputation going forward.

### Outliers

GDP and percentage expenditure are heavily right-skewed. I use the IQR method for
outlier detection because it is robust to the influence of the outliers
themselves and works well for skewed distributions. I compare two handling
approaches: removing outlier rows, and capping them to the IQR boundaries.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.boxplot(data=life_exp_df, y='GDP', ax=axes[0], width=0.4,
    boxprops=dict(facecolor='skyblue', edgecolor='black'),
    whiskerprops=dict(color='black'), capprops=dict(color='black'),
    medianprops=dict(color='red', linewidth=2))
axes[0].set_title('GDP (Before Outlier Removal)')
axes[0].set_xlabel(''); axes[0].set_ylabel('GDP')
axes[0].grid(True, linestyle='--', alpha=0.5)

sns.boxplot(data=life_exp_df, y='percentage expenditure', ax=axes[1], width=0.4,
    boxprops=dict(facecolor='lightgreen', edgecolor='black'),
    whiskerprops=dict(color='black'), capprops=dict(color='black'),
    medianprops=dict(color='red', linewidth=2))
axes[1].set_title('Percentage Expenditure (Before Outlier Removal)')
axes[1].set_xlabel(''); axes[1].set_ylabel('Percentage Expenditure')
axes[1].grid(True, linestyle='--', alpha=0.5)

legend_elements = [
    Patch(facecolor='skyblue', edgecolor='black', label='Box (IQR)'),
    Line2D([0], [0], color='red', linewidth=2, label='Median'),
    Line2D([0], [0], color='black', linewidth=1, label='Whiskers'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='black', markersize=8, label='Outliers')]
fig.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5, -0.02), ncol=4, frameon=True)
plt.tight_layout()
plt.savefig('../images/outliers_before.png', dpi=120, bbox_inches='tight')
plt.show()

#### Option A: removing outliers

In [ ]:
print(f"Original dataset size: {len(life_exp_df)}")

outlier_info = []
for col in ['GDP', 'percentage expenditure']:
    df_temp = life_exp_df.copy()
    Q1 = df_temp[col].quantile(0.25)
    Q3 = df_temp[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = ((df_temp[col] < lower_bound) | (df_temp[col] > upper_bound)).sum()
    outlier_info.append({'Column': col,
                         'Outliers (if removed alone)': outliers,
                         'Percentage': (outliers / len(life_exp_df)) * 100})

outlier_df_independent = pd.DataFrame(outlier_info)

life_df_no_outliers = life_exp_df.copy()
for col in ['GDP', 'percentage expenditure']:
    Q1 = life_df_no_outliers[col].quantile(0.25)
    Q3 = life_df_no_outliers[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    life_df_no_outliers = life_df_no_outliers[
        (life_df_no_outliers[col] >= lower_bound) & (life_df_no_outliers[col] <= upper_bound)]

outlier_rows = len(life_exp_df) - len(life_df_no_outliers)
print(f"\nAfter removing outliers from both columns:")
print(f"Dataset size: {len(life_df_no_outliers)}")
print(f"Total rows removed: {outlier_rows} ({outlier_rows/len(life_exp_df)*100:.2f}%)\n")
outlier_df_independent

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.boxplot(data=life_df_no_outliers, y='GDP', ax=axes[0], width=0.45,
    boxprops=dict(facecolor='skyblue', edgecolor='black'),
    whiskerprops=dict(color='black'), capprops=dict(color='black'),
    medianprops=dict(color='red', linewidth=2))
axes[0].grid(True, linestyle='--', alpha=0.5)
axes[0].set_title('GDP (After Outlier Removal)')
axes[0].set_xlabel(''); axes[0].set_ylabel('GDP')

sns.boxplot(data=life_df_no_outliers, y='percentage expenditure', ax=axes[1], width=0.45,
    boxprops=dict(facecolor='lightgreen', edgecolor='black'),
    whiskerprops=dict(color='black'), capprops=dict(color='black'),
    medianprops=dict(color='red', linewidth=2))
axes[1].set_title('Percentage Expenditure (After Outlier Removal)')
axes[1].set_xlabel(''); axes[1].set_ylabel('Percentage Expenditure')
axes[1].grid(True, linestyle='--', alpha=0.5)

legend_elements = [
    Patch(facecolor='skyblue', edgecolor='black', label='Box (IQR)'),
    Line2D([0], [0], color='red', linewidth=2, label='Median'),
    Line2D([0], [0], color='black', linewidth=1, label='Whiskers'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='black', markersize=8, label='Outliers')]
fig.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5, -0.02), ncol=4, frameon=True)
plt.tight_layout()
plt.show()

#### Option B: capping outliers (chosen)

In [ ]:
life_df_capping = life_exp_df.copy()
capping_info = []
for col in ['GDP', 'percentage expenditure']:
    Q1 = life_df_capping[col].quantile(0.25)
    Q3 = life_df_capping[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers_low = (life_df_capping[col] < lower_bound).sum()
    outliers_high = (life_df_capping[col] > upper_bound).sum()
    total_outliers = outliers_low + outliers_high
    life_df_capping[col] = life_df_capping[col].clip(lower=lower_bound, upper=upper_bound)
    capping_info.append({'Column': col, 'Outliers Capped': total_outliers,
                         'Lower Bound': lower_bound, 'Upper Bound': upper_bound})

capping_df = pd.DataFrame(capping_info)
capping_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.boxplot(data=life_df_capping, y='GDP', ax=axes[0], width=0.45,
    boxprops=dict(facecolor='skyblue', edgecolor='black'),
    whiskerprops=dict(color='black'), capprops=dict(color='black'),
    medianprops=dict(color='red', linewidth=2))
axes[0].grid(True, linestyle='--', alpha=0.5)
axes[0].set_title('GDP (After Capping)')
axes[0].set_xlabel(''); axes[0].set_ylabel('GDP')

sns.boxplot(data=life_df_capping, y='percentage expenditure', ax=axes[1], width=0.45,
    boxprops=dict(facecolor='lightgreen', edgecolor='black'),
    whiskerprops=dict(color='black'), capprops=dict(color='black'),
    medianprops=dict(color='red', linewidth=2))
axes[1].set_title('Percentage Expenditure (After Capping)')
axes[1].set_xlabel(''); axes[1].set_ylabel('Percentage Expenditure')
axes[1].grid(True, linestyle='--', alpha=0.5)

legend_elements = [
    Patch(facecolor='skyblue', edgecolor='black', label='Box (IQR)'),
    Line2D([0], [0], color='red', linewidth=2, label='Median'),
    Line2D([0], [0], color='black', linewidth=1, label='Whiskers'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='black', markersize=8, label='Outliers (if any)')]
fig.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5, -0.02), ncol=4, frameon=True)
plt.tight_layout()
plt.savefig('../images/outliers_after_capping.png', dpi=120, bbox_inches='tight')
plt.show()

There were 365 GDP outliers and 389 percentage-expenditure outliers. Removing
them from both columns sequentially would drop 992 rows (about 34% of the data).
Capping preserved all 2,938 rows while limiting the influence of extreme values,
so I chose capping over removal to avoid losing roughly a third of the dataset
and potentially valuable information.

## 3. Feature engineering

I create five new features by combining and transforming existing variables to
capture more complex relationships: infant and adult survival rates, a
vaccination coverage index (average of Polio, Diphtheria, and Hepatitis B), an
education-income index (schooling times income composition), and health spending
per capita.

In [ ]:
life_df_new = life_df_capping

life_df_new['Infant Survival Rate'] = 1 - (life_df_new['infant deaths'] / 1000)
life_df_new['Adult Survival Rate'] = 1 - (life_df_new['Adult Mortality'] / 1000)
life_df_new['Vaccination Coverage Index'] = life_df_new[['Polio', 'Diphtheria ', 'Hepatitis B']].mean(axis=1)
life_df_new['Education Income Index '] = life_df_new['Schooling'] * life_df_new['Income composition of resources']
life_df_new['Health_Spending_per_Capita'] = life_df_new['percentage expenditure'] / life_df_new['Population']

life_df_new.head()

### Distribution comparison and correlations

I look at how five selected indicators (Adult Mortality, Schooling, Income
composition of resources, BMI, HIV/AIDS) differ between developed and developing
countries, then check their correlations with life expectancy.

In [ ]:
my_selected_variables = [
    'Adult Mortality', 'Schooling', 'Income composition of resources', ' HIV/AIDS', ' BMI ']

print("Selected Independent Variables I believe to influence life expectancy:\n")
for i in my_selected_variables:
    print(f"{i}")

numerical_columns = life_df_new.select_dtypes(include=['number'])
correlation_matrix = numerical_columns.corr()
all_correlations = correlation_matrix['Life expectancy '].sort_values(ascending=False)
print("\nTop 5 Positive Correlations with Life Expectancy:")
print(all_correlations.head(5))
print("\nTop 5 Negative Correlations with Life Expectancy:")
print(all_correlations.tail(5))

selected_variables = ['Adult Mortality', 'Schooling', 'Income composition of resources', ' BMI ', ' HIV/AIDS']

In [ ]:
# Summary statistics for comparison between developed and developing countries
for var in selected_variables:
    print(f"\n{var}:")
    print(life_df_new.groupby('Status')[var].describe()[['mean', 'std', 'min', 'max']])

In [ ]:
# Correlation heatmap to visualize relationships among variables
correlation_vars = ['Life expectancy '] + selected_variables
correlation_matrix = life_df_new[correlation_vars].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='Blues', cbar_kws={'label': 'Correlation Coefficient'})
plt.title('Correlation Heatmap of Life Expectancy and Selected Independent Variables', pad=20)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('../images/correlation_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
life_exp_correlations = correlation_matrix['Life expectancy '].drop('Life expectancy ')
life_exp_correlations = life_exp_correlations.sort_values(ascending=False)
print("Correlations with Life Expectancy\n")
print(life_exp_correlations)

strongest_positive = life_exp_correlations.idxmax()
strongest_positive_value = life_exp_correlations.max()
strongest_negative = life_exp_correlations.idxmin()
strongest_negative_value = life_exp_correlations.min()

print(f"\n The strong positive correlation is {strongest_positive} with a correlation coefficient of {strongest_positive_value:.3f}")
print(f"\n The strong negative correlation is {strongest_negative} with a correlation coefficient of {strongest_negative_value:.3f}")

Schooling shows the strongest positive correlation (about 0.75), followed by
income composition (about 0.72), which makes sense since education and economic
development enable better health choices and access to healthcare. Adult
Mortality shows the strongest negative correlation (about -0.70), with HIV/AIDS
at around -0.56. Notably, the engineered Education-Income Index reaches about
0.79, higher than either Schooling or Income composition alone, which shows the
value of feature engineering.

## 4. Encoding

I encode Status with LabelEncoder (Developing = 1, Developed = 0) so it can be
used by the models. I do not encode Country because it has 193 unique values,
which would create too many dummy variables.

In [ ]:
life_df_processed = life_df_new.copy()
label_encoder = LabelEncoder()
life_df_processed['Status_Encoded'] = label_encoder.fit_transform(life_df_processed['Status'])
life_df_processed.tail(10)

## 5. Model training and hyperparameter tuning

Here I make one change from a naive workflow: I split the data into train and
test **before** imputing and scaling, then put those steps inside an sklearn
`Pipeline` so they are fit on the training data only. This keeps test
information out of training and out of cross-validation.

I select all numerical features (excluding the target), build the feature matrix,
and remove the few rows where the target itself is missing.

In [ ]:
# Selecting numerical features for modeling (exclude target)
numerical_features = life_df_processed.select_dtypes(include=[np.number]).columns.tolist()
features = [col for col in numerical_features if col != 'Life expectancy ']

# Removing rows where the target is missing
data_clean = life_df_processed.dropna(subset=['Life expectancy '])
X = data_clean[features]
y = data_clean['Life expectancy ']

print(f"Total features for modeling: {len(features)}")
print(f"Final dataset: {len(X)} rows")

In [ ]:
# Split FIRST, then preprocess inside pipelines
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Train:', X_train.shape, '| Test:', X_test.shape)

I train four regressors, each tuned with 5-fold `GridSearchCV` on negative mean
squared error. Tree-based models use the raw features; KNN uses scaled features,
since it relies on distance calculations and is sensitive to feature scale.
Imputation (and scaling for KNN) live inside each model's pipeline.

In [ ]:
def make_pipeline(estimator, scale):
    steps = [('impute', KNNImputer(n_neighbors=5))]
    if scale:
        steps.append(('scale', StandardScaler()))
    steps.append(('model', estimator))
    return Pipeline(steps)

models = {
    'Decision Tree': {
        'estimator': DecisionTreeRegressor(random_state=42), 'scaled': False,
        'params': {'model__max_depth': [3, 5, 10, None]}},
    'Random Forest': {
        'estimator': RandomForestRegressor(random_state=42), 'scaled': False,
        'params': {'model__n_estimators': [50, 100, 200], 'model__max_depth': [10, 15, 20]}},
    'KNN': {
        'estimator': KNeighborsRegressor(), 'scaled': True,
        'params': {'model__n_neighbors': [3, 5, 7, 11], 'model__weights': ['uniform', 'distance'], 'model__p': [1, 2]}},
    'Gradient Boosting': {
        'estimator': GradientBoostingRegressor(random_state=42), 'scaled': False,
        'params': {'model__n_estimators': [50, 100, 200], 'model__learning_rate': [0.01, 0.1, 0.2]}},
}

In [ ]:
results, fitted = [], {}
for name, config in models.items():
    print(f"Training {name}...")
    grid = GridSearchCV(make_pipeline(config['estimator'], config['scaled']),
                        config['params'], cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
    grid.fit(X_train, y_train)
    fitted[name] = grid.best_estimator_
    y_pred_train = grid.predict(X_train)
    y_pred_test = grid.predict(X_test)
    results.append({
        'Model': name,
        'Best Params': grid.best_params_,
        'Train RMSE': np.sqrt(mean_squared_error(y_train, y_pred_train)),
        'Test RMSE': np.sqrt(mean_squared_error(y_test, y_pred_test)),
        'Test MAE': mean_absolute_error(y_test, y_pred_test),
        'Test R²': r2_score(y_test, y_pred_test)})

results_df = pd.DataFrame(results).sort_values('Test RMSE').reset_index(drop=True)
results_df

Random Forest is the best-performing model, with the lowest test RMSE and the
highest R². Its train error is lower than its test error, which shows some
overfitting, but it still generalizes well. The ensemble methods (Random Forest
and Gradient Boosting) outperform the single Decision Tree, which a single tree
cannot match.

## 6. Feature importance and model comparison

I extract feature importances from the best (Random Forest) model, then retrain
each model using only the top 5 features to weigh accuracy against simplicity.

In [ ]:
best_model_name = results_df.iloc[0]['Model']
best_pipeline = fitted[best_model_name]
best_model = best_pipeline.named_steps['model']
print(f"Best Model: {best_model_name}")

importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': best_model.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(8, 6))
plt.barh(importance_df['Feature'], importance_df['Importance'], color='blue')
plt.xlabel('Importance')
plt.title(f'Feature Importance - {best_model_name}')
plt.gca().invert_yaxis()
plt.grid(alpha=0.1, axis='x')
plt.tight_layout()
plt.savefig('../images/feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()

importance_df

In [ ]:
top_5_features = importance_df.head(5)['Feature'].tolist()
print(f"\nTop 5 Most Important Features:")
for i, feature in enumerate(top_5_features, 1):
    importance = importance_df[importance_df['Feature'] == feature]['Importance'].values[0]
    print(f"  {i}. {feature}: {importance:.4f}")

HIV/AIDS and Income composition of resources dominate, together accounting for
roughly three-quarters of the model's decision-making. The remaining top features
(Adult Mortality, Adult Survival Rate, BMI) contribute much smaller amounts.

This doesn't fully match the simple correlations from section 3: Adult Mortality
had the strongest correlation with life expectancy but ranks lower in importance,
while HIV/AIDS has a weaker correlation but the highest importance. Correlation
only measures linear, one-at-a-time relationships, whereas tree-based importance
captures non-linear effects and interactions between features.

### Full vs. reduced (top-5) model

In [ ]:
top_feature_results = []
for name, config in models.items():
    print(f"Retraining {name} with Top 5 Features...")
    params = {k.replace('model__', ''): v
              for k, v in results_df[results_df['Model'] == name]['Best Params'].iloc[0].items()}
    pipe = make_pipeline(config['estimator'].set_params(**params), config['scaled'])
    pipe.fit(X_train[top_5_features], y_train)
    y_pred_top = pipe.predict(X_test[top_5_features])
    top_feature_results.append({
        'Model': name, 'Features Used': 'Top 5',
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred_top)),
        'MAE': mean_absolute_error(y_test, y_pred_top),
        'R²': r2_score(y_test, y_pred_top)})

full_feature_results = results_df[['Model', 'Test RMSE', 'Test MAE', 'Test R²']].copy()
full_feature_results.columns = ['Model', 'RMSE', 'MAE', 'R²']
full_feature_results['Features Used'] = 'All Features'

comparison_df = pd.concat([
    full_feature_results[['Model', 'Features Used', 'RMSE', 'MAE', 'R²']],
    pd.DataFrame(top_feature_results)[['Model', 'Features Used', 'RMSE', 'MAE', 'R²']]
], ignore_index=True)

print("\nPERFORMANCE COMPARISON: FULL FEATURES VS TOP FEATURES")
comparison_df.sort_values(['Model', 'Features Used']).reset_index(drop=True)

In [ ]:
feature_reduction = (1 - len(top_5_features) / len(X_train.columns)) * 100
print(f"\nFeatures reduced by: {feature_reduction:.1f}%\n")
print("Performance Changes (Top 5 vs All Features):\n")
for model_name in comparison_df['Model'].unique():
    full_row = comparison_df[(comparison_df['Model'] == model_name) & (comparison_df['Features Used'] == 'All Features')].iloc[0]
    top_row = comparison_df[(comparison_df['Model'] == model_name) & (comparison_df['Features Used'] == 'Top 5')].iloc[0]
    print(f"{model_name}:")
    print(f"  RMSE change: {top_row['RMSE'] - full_row['RMSE']:+.3f}")
    print(f"  R² change:   {top_row['R²'] - full_row['R²']:+.3f}\n")

For most models, using only the top 5 features makes performance slightly
worse, since removing features limits what they can learn. KNN actually improves,
because it is sensitive to irrelevant or noisy features and keeping only the
strongest predictors makes it simpler and more focused.

Overall, the reduced model keeps almost all of the full model's predictive power
while using a fraction of the features, which is a worthwhile tradeoff when the
goal is a model that is easier to interpret, faster to train, and cheaper to
collect data for, as in a low-resource healthcare setting.

## Summary

- **Best model:** Random Forest, tuned with 5-fold cross-validation.
- **Top predictors:** HIV/AIDS prevalence and income composition of resources,
  which together drive most of the model's decisions.
- **Method choices:** KNN imputation (preserved the data distribution better than
  median), IQR capping instead of removal (to avoid losing ~34% of the data),
  and five engineered features, the strongest of which out-correlated its parts.
- **Leakage-free modeling:** all learned preprocessing is fit on the training set
  only, inside a pipeline.
- **Reduced model:** the top-5-feature version keeps nearly all the accuracy with
  far fewer inputs, a good simplicity/accuracy tradeoff.